In [7]:
import requests
import pandas as pd
import json
import os
import sys

if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    from kaggle_secrets import UserSecretsClient
    print('Seems like this notebook runs in Kaggle, we will not import anything additional')
else:
    try:
        from google.colab import userdata
        from google.colab import drive
        drive.mount('/content/drive')
        print('Seems like this notebook runs in Google Colab. If not - please check import sequence and change a code')
    except:
        from dotenv import load_dotenv
        load_dotenv()
        print('Seems like this notebook runs in local environment, loaded .env file')

Seems like this notebook runs in Kaggle, we will not import anything additional


# NASA Data Export
Important. On default it uses test run. If you don't need this - change a variable in the next cell:

In [11]:
is_test_run = True

In [8]:
# Credentials and path to file depending on an environment
if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    credentials = UserSecretsClient().get_secret("NASA_ACCESS_KEY")
    csv_file_path ='/kaggle/working/all_neos.csv'
    print('Imported credentials for Kaggle and path for it')
elif 'google' in sys.modules:
    credentials = userdata.get('NASA_ACCESS_KEY')
    csv_file_path = '/content/drive/MyDrive/temp_colab_data/all_neos.csv'
    print('Imported credentials for Google and path for it')
else:
    credentials = os.getenv('NASA_ACCESS_KEY')
    csv_file_path = '../data/nasa_data/all_neos.csv'
    print('Imported credentials for local use and path for it')

Imported credentials for Kaggle and path for it


In [13]:
# Setting up a batch size. I know this is not the best way to do it, but it works without any issues

url = 'https://api.nasa.gov/neo/rest/v1/neo/browse?api_key=' + credentials
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
else:
    print(f"Initial request failed with status code: {response.status_code}")
    sys.exit(0)

# Number of cycles based on a fact is this a test run
if is_test_run:
    cycle_count = 10
else:
    cycle_count = data['page']['total_pages']

# Now we are exporting all NEOs to a csv file
for _ in range(cycle_count):
    # We have to be able to stop our script at any point if it crashes or something and continue from where we left off
    try:
        if _ in pd.read_csv(csv_file_path, usecols=['page'])['page'].values:
            print('skipped',_)
            continue
    except:
        pass

    # Creating a link
    url = f'https://api.nasa.gov/neo/rest/v1/neo/browse?page={_}&size=20&api_key=' + credentials
    print('loading page ' + str(_))

    # It's just more understandable this way cmon
    response = requests.get(url)
    data = response.json()

    filtered_data = pd.DataFrame(data['near_earth_objects']).drop('links', axis=1) # Dropping the links column as a security concern
    filtered_data['page'] = _ #Adding a page column to the dataframe
    filtered_data = filtered_data[['id', 'neo_reference_id', 'name', 'designation', 'nasa_jpl_url',
       'absolute_magnitude_h', 'estimated_diameter',
       'is_potentially_hazardous_asteroid', 'close_approach_data',
       'orbital_data', 'is_sentry_object', 'page']] # Using fixed list of columns in a fixed order

    # Exporting the data
    if response.status_code == 200:
        if _ == 0: # If it's the first page, we write the header and creating a file basically
            filtered_data.to_csv(csv_file_path, mode='w', index=False, header=True)
        else:
            filtered_data.to_csv(csv_file_path, mode='a', index=False, header=False)
    else:
        print(f"Request failed with status code: {response.status_code}")
        break

skipped 0
skipped 1
skipped 2
skipped 3
skipped 4
skipped 5
skipped 6
skipped 7
skipped 8
skipped 9
